# Rollout Visualization

This notebook :

- Load original dataset and dataset with saved rollouts
- Visualize trajectories from expert and from rollouts

## Setup

1. Setup environment and paths
2. Import libraries and modules
3. Load configuration file
4. Load robot
5. Load validation TrajectoryDataset

In [1]:
%cd /workspace
# Autoreload for Jupyter to reflect changes in other modules without restarting the kernel
%reload_ext autoreload
%autoreload 2

import sys
import os

# Set PYTHONPATH for robofin
robofin_path = os.path.join(os.getcwd(), 'robofin')
current_pythonpath = os.environ.get('PYTHONPATH', '')
if robofin_path not in current_pythonpath:
    os.environ['PYTHONPATH'] = f"{robofin_path}:{current_pythonpath}" if current_pythonpath else robofin_path

# Also add to sys.path for immediate effect
if robofin_path not in sys.path:
    sys.path.insert(0, robofin_path)

import torch
import numpy as np
import yaml
from pathlib import Path
from torch.utils.data import DataLoader

# Import dataset and robot modules
from avoid_everything.data_loader import TrajectoryDataset, StateDataset
from avoid_everything.type_defs import DatasetType
from robofin.robots import Robot
from utils import visualization
import viz_client

print("✓ All imports successful!")

# Load configuration from evaluation.yaml
config_path = "/workspace/model_configs/evaluation.yaml"
with open(config_path, 'r') as f:
    cfg = yaml.safe_load(f)

# Initialize robot
urdf_path = cfg['shared_parameters']['urdf_path']
robot = Robot(urdf_path, device='cpu')
print(f"✓ Robot loaded from URDF: {urdf_path}")

# Load val dataset
# Load validation dataset
val_dataset = TrajectoryDataset.load_from_directory(
    robot=robot,
    directory=cfg['data_module_parameters']['data_dir'],
    dataset_type=DatasetType.VAL,
    trajectory_key=cfg['data_module_parameters']['val_trajectory_key'],
    num_robot_points=cfg['shared_parameters']['num_robot_points'],
    num_obstacle_points=cfg['data_module_parameters']['num_obstacle_points'],
    num_target_points=cfg['data_module_parameters']['num_target_points'],
    random_scale=0.0  # No noise for validation
)
print(f"✓ Validation dataset loaded with {len(val_dataset)} samples")

/usr/local/lib/python3.10/dist-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


/workspace


/usr/local/lib/python3.10/dist-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


✓ All imports successful!
✓ Robot loaded from URDF: /workspace/assets/panda/panda_spheres.urdf
✓ Validation dataset loaded with 10034 samples


(Disconnect and) Connect to visualization server

In [2]:
# Connect to visualization server
viz_client.shutdown()
viz_client.connect(str(robot.urdf_path))
viz_client.publish_joints(joints=robot.neutral_config_dict)

Not connected to viz_server
viz_server not running — starting…
Connected to viz_server


# Visualize expert trajectories

Visualize single expert trajectory,
2 animation modalities:
+ "interpolation": for smooth linear interpolation of the trajectory waypoints 
+ "discrete": for discrete steps between waypoints

In [3]:
animation_modality = "discrete"  # "interpolation" or "discrete"
sample_0 = val_dataset[2]

print(f"Animating expert trajectory...")
visualization.visualize_expert_trajectory(
    robot=robot,
    sample=sample_0,
    mode =animation_modality,
)

print("✓ Expert trajectory visualization started!")

Animating expert trajectory...
Original trajectory length: 60 states
Trimmed trajectory length: 41 states
Trajectory length: 41
The trajectory is collision-free
✓ Expert trajectory visualization started!


Interactive slider to select different samples from the dataset and visualize their expert trajectories:

In [4]:
visualization.visualize_expert_trajectory_slider(
    dataset=val_dataset,
    robot=robot,
    animation_mode=animation_modality,
)

IntSlider(value=0, continuous_update=False, description='Trajectory Index:', max=10033, style=SliderStyle(desc…

Output()

Interactive slider to visualize different states/steps along the trajectory:

In [5]:
sample = val_dataset[2]
visualization.visualize_states_in_trajectories_slider(sample, robot, expert=True)

IntSlider(value=0, continuous_update=False, description='Timestep Index:', max=40)

Output()

In [7]:
visualization.visualize_states_nested_slider(
    dataset=val_dataset,
    robot=robot,
    expert=True,
)

# Visualize saved rollout trajectories

Load FailureAnalyzer and saved rollouts dataset

In [11]:
from utils.failure_analysis import FailureAnalyzer
from utils.failure_analysis import SavedTrajectoryDataset

saved_trajectory_db = "/workspace/datasets/failed_trajectories/failed_trajectories_20260223_095140.hdf5"
analyzer = FailureAnalyzer(load_existing=saved_trajectory_db,
                           original_dataset=val_dataset)

if not hasattr(analyzer, 'num_saved'): # if no metadata, set default
    analyzer.num_saved = 30
saved_trajectory_val_dataset = SavedTrajectoryDataset(analyzer)

✓ Loaded existing FailureAnalyzer from: /workspace/datasets/failed_trajectories/failed_trajectories_20260223_095140.hdf5


Load only saved trajectory and trajectory data

In [12]:
idx = 1
trajectory_i = analyzer.load(idx, load_scene=False)
print(f"Loaded trajectory {idx}")
for key, value in trajectory_i.items():
    if hasattr(value, 'shape') and len(value.shape) > 1:
        print(f"- {key}: {value.shape}")
    else:
        print(f"- {key}: {value}")

Loaded trajectory 1
- trajectory: (71, 7)
- has_collision: True
- has_reaching_success: True
- orientation_error: 1.0032720565795898
- pidx: 26
- position_error: 0.003868142608553171


Load both trajectory and sample from original dataset

In [8]:
idx = 1
trajectory_i = analyzer.load(idx, load_scene=True)
print(f"Loaded trajectory {idx}")
for key, value in trajectory_i.items():
    if hasattr(value, 'shape') and len(value.shape) > 1:
        print(f"- {key}: {value.shape}")
    else:
        print(f"- {key}: {value}")

Loaded trajectory 1
- trajectory: (71, 7)
- has_collision: True
- has_reaching_success: True
- orientation_error: 1.0032720565795898
- pidx: 26
- position_error: 0.003868142608553171
- target_position: tensor([0.6224, 0.1077, 0.3060])
- target_orientation: torch.Size([3, 3])
- cuboid_dims: torch.Size([8, 3])
- cuboid_centers: torch.Size([8, 3])
- cuboid_quats: torch.Size([8, 4])
- cylinder_radii: torch.Size([0, 1])
- cylinder_heights: torch.Size([0, 1])
- cylinder_centers: torch.Size([0, 3])
- cylinder_quats: torch.Size([0, 4])
- point_cloud: torch.Size([6272, 3])
- point_cloud_labels: torch.Size([6272, 1])
- configuration: tensor([-0.4754,  0.4030,  0.3681,  0.1392,  0.3373,  0.1027, -0.3195])
- expert: torch.Size([60, 7])


Visualize trajectory and the ghost robot configuration of the first collision in the rollout

In [13]:
idx = 3
sample_i = analyzer.load(idx, load_scene=True)
print(f"Visualizing trajectory {idx}")

visualization.visualize_trajectory(
    robot=robot,
    sample=sample_i,
    trajectory=sample_i["trajectory"],
    mode=animation_modality,
    normalized=False,
)

Visualizing trajectory 3
Trajectory length: 11
⚠ Collision detected at timestep(s): [7, 8, 9, 10]


Slider to select trajectory from the dataset and visualize it

In [14]:
visualization.visualize_saved_trajectory_slider(
    dataset=saved_trajectory_val_dataset,
    robot=robot,
    animation_mode=animation_modality,
)

IntSlider(value=0, continuous_update=False, description='Trajectory Index:', max=29, style=SliderStyle(descrip…

Output()

Nested slider:
- Outer slider to select the trajectory
- Inner slider to select the timestep within the trajectory

In [16]:
# Nested slider for saved rollout trajectories
visualization.visualize_states_nested_slider(
    dataset=saved_trajectory_val_dataset,
    robot=robot,
)

Nested slider with trajectory animations

In [15]:
# Nested slider for saved rollout trajectories
visualization.visualize_states_nested_slider(
    dataset=saved_trajectory_val_dataset,
    robot=robot,
    highlight_collisions=True,
    animate_traj=True,
    # mode="interpolate" # TODO: mode interpolate for slider (currently not supported)
)